# Ardook grant fit: iterating on the similarity baseline

Starts from the baseline in `grant_similarity_baseline.ipynb` and changes one thing per round, based on what the previous round got wrong.

| Round | Change | Why |
|---|---|---|
| 0 | Similarity to a full Ardook description | Starting point |
| 1 | Theme-only and multiple short queries | The full description mixes geography into the score and penalizes US grants |
| 2 | Train a classifier on embeddings | Round 0 never used the 2,400 training grants |
| 3 | Similarity to a bank of criterion queries, then a classifier | Keeps the features explainable |
| 4 | Predict theme, geography and entity separately, then apply the label rule | Similarity alone misses eligibility rules |
| 5 | Fine-tune DistilBERT end to end | The standard Hugging Face approach |

**Fair comparison:** every method uses the same data. Settings and the final pick use validation only, and the test split is only used for reporting.

**How to run (Colab):** Runtime → Run all. Fine-tuning is much faster with a GPU. No files to upload.

> Synthetic-only results. The label rule that generated this data is the same one round 4 applies, so high scores do not prove the method works on real grants.

In [1]:
%pip -q install "sentence-transformers>=3.0" "transformers>=4.46" datasets accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc, os, random, re, shutil, time, warnings
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Apple MPS crashes on these models
print("device:", DEVICE)

LABELS = ["Low", "Moderate", "High"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

/private/tmp/claude-501/-Users-giapdo-Desktop-Ardook-proj/a4319d0f-99c3-4529-8b9d-efc359734cfd/scratchpad/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1105 11:31:15.039000 11042 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: cpu


## Data

Same generator and seeds as the baseline notebook, so the grants are identical.

In [3]:
THEMES = {
    "good": [
        "economic empowerment of women", "employment for mothers of children with disabilities",
        "disability inclusion and inclusive employment", "support for family caregivers of people with disabilities",
        "flexible, home-based work for women", "social enterprises that create decent jobs for vulnerable groups",
        "livelihoods for single mothers", "women-led social enterprises with measurable social impact",
        "inclusive businesses that employ marginalized women", "reducing inequality through dignified employment (SDG 8 and SDG 10)",
        "economic independence for caregivers", "income generation for women who cannot work outside the home",
        "job creation for disadvantaged women through social entrepreneurship", "families raising children with special needs",
        "women's livelihoods and income security", "work-from-home opportunities for parents of disabled children",
    ],
    "partial": [
        "growth of small and medium-sized enterprises", "women entrepreneurs in any sector",
        "green and eco-friendly small businesses", "community development projects",
        "youth employment and skills training", "service-sector small businesses",
        "digitalization of micro-enterprises", "local economic development in rural regions",
        "early-stage startups with a social mission", "circular economy and waste reduction",
        "civil society capacity building", "tourism and hospitality businesses",
    ],
    "bad": [
        "artificial intelligence and machine learning research", "deep-tech hardware startups",
        "biomedical and clinical research", "space and satellite technology",
        "quantum computing", "agricultural biotechnology R&D",
        "cybersecurity tools", "fintech and blockchain platforms",
        "contemporary art and cultural heritage", "academic STEM research at universities",
        "generative AI products", "semiconductor design",
        "climate science modelling", "battery and energy storage technology",
    ],
}
GEOS = {
    "good": [
        "organizations based in Armenia", "applicants from Eastern Partnership countries (Armenia, Azerbaijan, Georgia, Moldova, Ukraine)",
        "applicants worldwide", "organizations operating in the South Caucasus",
        "social enterprises registered in Armenia", "applicants from any country",
        "organizations in EU Neighbourhood East countries", "US-based organizations",
        "organizations registered in the United States", "applicants from low- and middle-income countries",
        "Armenian and international organizations", "organizations based in Yerevan or other Armenian regions",
    ],
    "partial": [
        "organizations registered and operating in Georgia", "applicants based in Tbilisi",
        "Georgian social enterprises", "organizations active in the regions of Georgia (the country)",
    ],
    "bad": [
        "organizations based in India", "Kenyan-registered companies", "residents of Brazil",
        "Canadian citizens and permanent residents", "UK-registered charities", "organizations in the Philippines",
        "applicants based in Nigeria", "Australian small businesses", "organizations in Southeast Asia",
        "applicants from Latin America", "Indonesian startups", "residents of Ontario",
    ],
}
ENTITIES = {
    "good": [
        "both for-profit companies and nonprofits are eligible", "social enterprises of any legal form may apply",
        "registered businesses with at least one year of operations may apply", "small businesses with paying customers are eligible",
        "early-stage and growth-stage ventures are welcome", "women-led enterprises are encouraged to apply",
        "organizations with a proven, revenue-generating model are eligible", "any legally registered organization may apply",
    ],
    "partial": [
        "only registered nonprofits / NGOs may apply", "companies must be less than 5 years old",
        "businesses must be at least 51% owned by women", "applicants must have annual revenue below $250,000",
        "only legally registered charities are eligible", "the organization must have been registered after 2021",
    ],
    "bad": [
        "applicants must demonstrate deep-tech innovation at TRL 4 or above", "the lead applicant must be a university or research institute",
        "projects must build an AI or software product", "a PhD-level principal investigator is required",
        "applicants must hold patented technology", "only hardware startups with a working prototype qualify",
    ],
}
FUNDER_PREFIX = ["Northbridge", "Silver Linden", "Ararat Horizons", "Harbor Light", "Open Field", "Cedar & Stone",
                 "Kestrel", "Brightwater", "Sevan Valley", "Lumen", "Tidewell", "Granite Peak",
                 "Willowmere", "Amber Coast", "Highland Commons", "Juniper", "Starling", "Riverbend", "Copperleaf", "Meadowgate"]
FUNDER_SUFFIX = ["Foundation", "Fund", "Trust", "Initiative", "Partnership", "Alliance"]
TYPES = ["grant", "microgrant", "prize", "accelerator", "fellowship", "challenge", "award", "seed fund"]
FILLERS = ["Applications are reviewed on a rolling basis.", "Shortlisted applicants will be invited to an interview.",
           "Selected teams receive mentoring in addition to funding.", "Reporting is required every six months.",
           "Applications must be submitted in English.", "Joint applications from consortia are accepted.",
           "Information sessions are held online before the deadline.", "Previous grantees may reapply."]

# Document templates. {theme}/{geo}/{entity} are filled from the pools above.
TEMPLATES = [
    "The {funder} invites applications for its {program}. The program supports projects focused on {theme}. Eligible applicants: {geo}. Note that {entity}. Awards of up to {amount} are available. {filler}",
    "{program} ({type}). Amount: {amount}. Deadline: {deadline}. Core requirements: {theme}; {geo}; {entity}.",
    "Call for proposals: {program}. Focus area: {theme}. Who can apply: {geo}. Eligibility: {entity}. Funding: {amount}. Deadline: {deadline}.",
    "The {program} backs work on {theme}. The call is open to {geo}, and {entity}. {filler} {extra}",
    "{funder} announces {amount} in funding through its {type} for {theme}. Applicants must be {geo}. Important: {entity}. {extra}",
    "Opportunity: {program}. We fund {theme}. {filler} Open to {geo}; {entity}. Award size: {amount}.",
    # --- below this line: test-only templates ---
    "About the opportunity: {program} is a {type} for initiatives in {theme}. Geographic scope: {geo}. Eligibility notes: {entity}. Up to {amount} per award. {extra}",
    "Funding priorities: {theme}. Applications are accepted from {geo}. {entity_cap}. Grant size: {amount}. Closing date: {deadline}. {filler}",
]
N_TRAIN_TEMPLATES = 6

def split_pool(pool, split):
    # Every 4th phrase is held out for the test split so test wording is partly unseen.
    return [p for i, p in enumerate(pool) if (i % 4 == 3) == (split == "test")]

def fit_label(theme, geo, entity):
    keys = (theme, geo, entity)
    if "bad" in keys:
        return "Low"
    if keys == ("good", "good", "good"):
        return "High"
    return "Moderate"

def make_grant(rng, split, theme, geo, entity):
    pick = lambda pool: rng.choice(split_pool(pool, split))
    funder = f"{pick(FUNDER_PREFIX)} {rng.choice(FUNDER_SUFFIX)}"
    gtype = pick(TYPES)
    # Hard negatives / distractors: an extra theme mention that does not change the label.
    if theme == "bad" and rng.random() < 0.4:
        extra = f"Priority is given to teams that also address {pick(THEMES['good'])}."
    elif rng.random() < 0.3:
        extra = f"Projects related to {pick(THEMES['partial'])} are also welcome."
    else:
        extra = ""
    templates = TEMPLATES[:N_TRAIN_TEMPLATES] if split != "test" else TEMPLATES[N_TRAIN_TEMPLATES:]
    ent = pick(ENTITIES[entity])
    text = rng.choice(templates).format(
        funder=funder, program=f"{funder} {gtype.title()}", type=gtype,
        theme=pick(THEMES[theme]), geo=pick(GEOS[geo]), entity=ent, entity_cap=ent[0].upper() + ent[1:],
        amount=f"{rng.choice(['$', '€'])}{rng.choice([2, 5, 10, 15, 25, 50, 100, 250]) * 1000:,}",
        deadline=f"{rng.randint(1, 28)} {rng.choice(['January', 'March', 'May', 'July', 'September', 'November'])} {rng.choice([2026, 2027])}",
        filler=rng.choice(FILLERS), extra=extra)
    return re.sub(r"\s+", " ", text).strip()

def generate(n_per_class, split, seed, noise=0.0):
    rng = random.Random(seed)
    counts = {l: 0 for l in LABELS}
    rows, seen = [], set()
    while min(counts.values()) < n_per_class:
        attrs = [rng.choice(["good", "partial", "bad"]) for _ in range(3)]
        label = fit_label(*attrs)
        if counts[label] >= n_per_class:
            continue
        text = make_grant(rng, split, *attrs)
        if text in seen:
            continue
        seen.add(text)
        counts[label] += 1
        rows.append({"text": text, "label": label, "theme": attrs[0], "geo": attrs[1], "entity": attrs[2]})
    df = pd.DataFrame(rows)
    if noise:
        # Flip a fraction of labels to a neighbouring class (Low<->Moderate<->High).
        flip = rng.sample(range(len(df)), int(noise * len(df)))
        for i in flip:
            j = LABEL2ID[df.at[i, "label"]]
            df.at[i, "label"] = LABELS[1] if j != 1 else rng.choice([LABELS[0], LABELS[2]])
    df["label_id"] = df["label"].map(LABEL2ID)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

In [4]:
N_TRAIN, N_VAL, N_TEST = 800, 100, 100   # per class (80/10/10)
train_df = generate(N_TRAIN, "train", seed=1)
val_df = generate(N_VAL, "val", seed=2)
test_df = generate(N_TEST, "test", seed=3)
y_train, y_val, y_test = train_df.label_id.values, val_df.label_id.values, test_df.label_id.values
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} n={len(df):4d}", df.label.value_counts().to_dict())

train n=2400 {'Moderate': 800, 'Low': 800, 'High': 800}
val   n= 300 {'Low': 100, 'High': 100, 'Moderate': 100}
test  n= 300 {'Low': 100, 'High': 100, 'Moderate': 100}


## Helpers

The queries below are summarized from `docs/ardook-profile.md`, the source of truth; update them together. The criterion queries were written generically and do not copy the test phrases.

In [5]:
ARDOOK_FULL = (
    "Ardook is a revenue-generating, women-led social enterprise in Yerevan, Armenia, expanding to Tbilisi, Georgia. "
    "It provides sustainable, home-based employment for mothers of children with disabilities, many of them single mothers, "
    "through a laundry and ironing service. Its focus is women's economic empowerment, disability inclusion, caregivers, "
    "inclusive employment, decent work, and livelihoods."
)
ARDOOK_THEME = (
    "A women-led social enterprise providing home-based employment for mothers of children with disabilities. "
    "Women's economic empowerment, disability inclusion, caregivers, inclusive employment, decent work, and livelihoods."
)
THEME_QUERIES = [
    "jobs for mothers of children with disabilities", "disability inclusion", "support for caregivers",
    "women's economic empowerment", "home-based flexible work", "social enterprise creating decent jobs",
    "livelihoods for single mothers",
]
CRITERION_QUERIES = THEME_QUERIES + [
    "small business growth", "women entrepreneurs", "artificial intelligence and technology research", "scientific research",
    "open to organizations in Armenia", "open to applicants from any country", "open to the Caucasus region",
    "open to organizations in the United States", "open only to organizations in Georgia",
    "restricted to organizations in one other country",
    "open to any type of organization", "only nonprofit organizations can apply", "limits on company age or revenue",
    "requires women's ownership", "requires a university or research institution", "requires deep technology or patents",
]
BGE_INSTRUCTION = "Represent this sentence for searching relevant passages: "

results, predictions = [], {}

def to_class(scores, low_cut, high_cut):
    return np.where(scores >= high_cut, LABEL2ID["High"], np.where(scores >= low_cut, LABEL2ID["Moderate"], LABEL2ID["Low"]))

def tune_cutoffs(scores, y):
    cands = np.unique(np.quantile(scores, np.linspace(0.02, 0.98, 49)))
    best, best_f1 = (cands[0], cands[-1]), -1
    for i, lo in enumerate(cands):
        for hi in cands[i + 1:]:
            f = f1_score(y, to_class(scores, lo, hi), average="macro", zero_division=0)
            if f > best_f1:
                best, best_f1 = (lo, hi), f
    return best

def f1_interval(y_true, y_pred, n=1000):
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(y_true), (n, len(y_true)))
    vals = [f1_score(y_true[i], y_pred[i], average="macro", zero_division=0) for i in idx]
    return np.percentile(vals, [2.5, 97.5])

def record(round_no, name, val_pred, test_pred, seconds):
    p, r, f, _ = precision_recall_fscore_support(y_test, test_pred, average="macro", zero_division=0)
    lo, hi = f1_interval(y_test, np.asarray(test_pred))
    val_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
    results.append({"round": round_no, "method": name, "val_f1": val_f1, "accuracy": accuracy_score(y_test, test_pred),
                    "precision": p, "recall": r, "f1": f, "f1_95ci": f"{lo:.3f}-{hi:.3f}", "seconds": round(seconds, 1)})
    predictions[name] = np.asarray(test_pred)
    print(f"R{round_no} {name:48s} val={val_f1:.3f} | acc={results[-1]['accuracy']:.3f} P={p:.3f} R={r:.3f} F1={f:.3f} [{lo:.3f}-{hi:.3f}] ({seconds:.0f}s)")

def record_similarity(round_no, name, val_s, test_s, seconds):
    assert np.isfinite(val_s).all() and np.isfinite(test_s).all(), f"{name} returned NaN/inf scores"
    lo, hi = tune_cutoffs(val_s, y_val)
    record(round_no, name, to_class(val_s, lo, hi), to_class(test_s, lo, hi), seconds)

def fit_logreg(X_tr, y_tr, X_va, y_va):
    # Pick regularization strength C on the validation set.
    best = None
    for C in [0.1, 1, 10, 100]:
        clf = LogisticRegression(C=C, max_iter=3000).fit(X_tr, y_tr)
        score = f1_score(y_va, clf.predict(X_va), average="macro", zero_division=0)
        if best is None or score > best[0]:
            best = (score, clf)
    return best[1]

In [6]:
# Embed everything once per model.
EMB = {}
for short, name, instruction in [("MiniLM", "sentence-transformers/all-MiniLM-L6-v2", ""),
                                 ("bge-small", "BAAI/bge-small-en-v1.5", BGE_INSTRUCTION)]:
    t = time.time()
    model = SentenceTransformer(name, device=DEVICE)
    enc = lambda texts: model.encode(texts, batch_size=64, normalize_embeddings=True)
    q = lambda texts: enc([instruction + x for x in texts])
    EMB[short] = {"train": enc(train_df.text.tolist()), "val": enc(val_df.text.tolist()), "test": enc(test_df.text.tolist()),
                  "full": q([ARDOOK_FULL]), "theme": q([ARDOOK_THEME]), "theme_bank": q(THEME_QUERIES),
                  "criteria": q(CRITERION_QUERIES), "seconds": time.time() - t}
    del model; gc.collect()
    print(short, "embedded in", round(EMB[short]["seconds"]), "s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14321.20it/s]

MiniLM embedded in 18 s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19206.28it/s]

bge-small embedded in 24 s


## Round 0 and 1: similarity with better queries

In [7]:
for short, E in EMB.items():
    t = time.time()
    record_similarity(0, f"{short}: full Ardook description", (E["val"] @ E["full"].T).ravel(), (E["test"] @ E["full"].T).ravel(), time.time() - t)
    t = time.time()
    record_similarity(1, f"{short}: theme-only description", (E["val"] @ E["theme"].T).ravel(), (E["test"] @ E["theme"].T).ravel(), time.time() - t)
    t = time.time()
    record_similarity(1, f"{short}: best of 7 short queries", (E["val"] @ E["theme_bank"].T).max(1), (E["test"] @ E["theme_bank"].T).max(1), time.time() - t)

R0 MiniLM: full Ardook description                  val=0.452 | acc=0.560 P=0.615 R=0.560 F1=0.548 [0.490-0.601] (0s)


R1 MiniLM: theme-only description                   val=0.596 | acc=0.460 P=0.429 R=0.460 F1=0.430 [0.379-0.476] (0s)


R1 MiniLM: best of 7 short queries                  val=0.568 | acc=0.463 P=0.424 R=0.463 F1=0.427 [0.375-0.476] (0s)


R0 bge-small: full Ardook description               val=0.512 | acc=0.507 P=0.489 R=0.507 F1=0.496 [0.443-0.544] (0s)


R1 bge-small: theme-only description                val=0.579 | acc=0.510 P=0.478 R=0.510 F1=0.487 [0.432-0.539] (0s)


R1 bge-small: best of 7 short queries               val=0.585 | acc=0.500 P=0.469 R=0.500 F1=0.476 [0.422-0.530] (0s)


## Round 2: train a classifier

In [8]:
# TF-IDF + classifier shows whether the embeddings add anything over plain word counts.
t = time.time()
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True).fit(train_df.text)
Xtr, Xva, Xte = tfidf.transform(train_df.text), tfidf.transform(val_df.text), tfidf.transform(test_df.text)
clf = fit_logreg(Xtr, y_train, Xva, y_val)
record(2, "TF-IDF + LogReg", clf.predict(Xva), clf.predict(Xte), time.time() - t)

for short, E in EMB.items():
    t = time.time()
    clf = fit_logreg(E["train"], y_train, E["val"], y_val)
    record(2, f"{short} embeddings + LogReg", clf.predict(E["val"]), clf.predict(E["test"]), time.time() - t)

R2 TF-IDF + LogReg                                  val=0.997 | acc=0.413 P=0.381 R=0.413 F1=0.360 [0.314-0.406] (0s)


R2 MiniLM embeddings + LogReg                       val=0.967 | acc=0.567 P=0.582 R=0.567 F1=0.560 [0.508-0.612] (0s)


R2 bge-small embeddings + LogReg                    val=0.973 | acc=0.680 P=0.694 R=0.680 F1=0.684 [0.633-0.735] (0s)


## Round 3: criterion similarity features + classifier

Each grant becomes a short list of scores, one per plain-English question such as "only nonprofit organizations can apply". A classifier then learns how to weigh them, so its coefficients can be read directly.

In [9]:
for short, E in EMB.items():
    t = time.time()
    F = {s: E[s] @ E["criteria"].T for s in ["train", "val", "test"]}
    clf = fit_logreg(F["train"], y_train, F["val"], y_val)
    record(3, f"{short}: {len(CRITERION_QUERIES)} criterion scores + LogReg", clf.predict(F["val"]), clf.predict(F["test"]), time.time() - t)

R3 MiniLM: 23 criterion scores + LogReg             val=0.823 | acc=0.647 P=0.657 R=0.647 F1=0.645 [0.591-0.696] (0s)


R3 bge-small: 23 criterion scores + LogReg          val=0.813 | acc=0.740 P=0.757 R=0.740 F1=0.740 [0.689-0.787] (0s)


## Round 4: predict each criterion separately

Three classifiers predict `theme`, `geo` and `entity`. The label rule then combines them: any `bad` → Low, all `good` → High, otherwise Moderate. This also produces a reason for each prediction, which a single score can't give.

In [10]:
criterion_accuracy = {}
for short, E in EMB.items():
    t = time.time()
    val_attrs, test_attrs, acc = {}, {}, {}
    for col in ["theme", "geo", "entity"]:
        clf = fit_logreg(E["train"], train_df[col], E["val"], val_df[col])
        val_attrs[col], test_attrs[col] = clf.predict(E["val"]), clf.predict(E["test"])
        acc[col] = accuracy_score(test_df[col], test_attrs[col])
    combine = lambda a: [LABEL2ID[fit_label(*x)] for x in zip(a["theme"], a["geo"], a["entity"])]
    criterion_accuracy[short] = acc
    record(4, f"{short}: per-criterion classifiers + rule", combine(val_attrs), combine(test_attrs), time.time() - t)
print("Test accuracy of each criterion:", {k: {c: round(v, 3) for c, v in a.items()} for k, a in criterion_accuracy.items()})

R4 MiniLM: per-criterion classifiers + rule         val=0.987 | acc=0.577 P=0.626 R=0.577 F1=0.568 [0.512-0.620] (0s)


R4 bge-small: per-criterion classifiers + rule      val=0.997 | acc=0.647 P=0.670 R=0.647 F1=0.649 [0.590-0.702] (0s)
Test accuracy of each criterion: {'MiniLM': {'theme': 0.817, 'geo': 0.717, 'entity': 0.55}, 'bge-small': {'theme': 0.86, 'geo': 0.813, 'entity': 0.517}}


## Round 5: fine-tune DistilBERT

In [11]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

FT_NAME, EPOCHS, MAX_LEN = "distilbert-base-uncased", 3, 128
t = time.time()
tok = AutoTokenizer.from_pretrained(FT_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    FT_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)

def to_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label_id"]].rename(columns={"label_id": "labels"}), preserve_index=False)
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])

def macro_f1(eval_pred):
    logits, labels = eval_pred
    return {"f1_macro": f1_score(labels, logits.argmax(-1), average="macro", zero_division=0)}

out_dir = "/tmp/ardook_distilbert"
args = TrainingArguments(
    output_dir=out_dir, num_train_epochs=EPOCHS, learning_rate=5e-5, weight_decay=0.01,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    use_cpu=DEVICE == "cpu", logging_steps=100, report_to="none", seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train_df), eval_dataset=to_dataset(val_df),
                  processing_class=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=macro_f1)
trainer.train()
val_logits = trainer.predict(to_dataset(val_df)).predictions
test_logits = trainer.predict(to_dataset(test_df)).predictions
if np.isfinite(val_logits).all() and np.isfinite(test_logits).all():
    record(5, "DistilBERT fine-tuned", val_logits.argmax(-1), test_logits.argmax(-1), time.time() - t)
else:
    print("DistilBERT returned NaN/inf on this machine, so it has no score here. Rerun with a GPU.")
del trainer, model; gc.collect(); shutil.rmtree(out_dir, ignore_errors=True)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8633.28it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:  42%|████▏     | 1000/2400 [00:00<00:00, 8085.65 examples/s]

Map:  83%|████████▎ | 2000/2400 [00:00<00:00, 8924.15 examples/s]

... (14 outputs left out)

Map: 100%|██████████| 300/300 [00:00<00:00, 4446.70 examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map: 100%|██████████| 300/300 [00:00<00:00, 6454.70 examples/s]

R5 DistilBERT fine-tuned                            val=1.000 | acc=0.707 P=0.774 R=0.707 F1=0.689 [0.640-0.738] (345s)


## Results by round

In [12]:
res = pd.DataFrame(results)
os.makedirs("results", exist_ok=True)
res.to_csv("results/iteration_comparison.csv", index=False)
display(Markdown(res.round(3).to_markdown(index=False)))
best_by_round = res.loc[res.groupby("round").val_f1.idxmax()]
print("Best method of each round, chosen on validation:")
display(Markdown(best_by_round[["round", "method", "val_f1", "accuracy", "precision", "recall", "f1", "f1_95ci"]].round(3).to_markdown(index=False)))

|   round | method                                      |   val_f1 |   accuracy |   precision |   recall |    f1 | f1_95ci     |   seconds |
|--------:|:--------------------------------------------|---------:|-----------:|------------:|---------:|------:|:------------|----------:|
|       0 | MiniLM: full Ardook description             |    0.452 |      0.56  |       0.615 |    0.56  | 0.548 | 0.490-0.601 |       0   |
|       1 | MiniLM: theme-only description              |    0.596 |      0.46  |       0.429 |    0.46  | 0.43  | 0.379-0.476 |       0   |
|       1 | MiniLM: best of 7 short queries             |    0.568 |      0.463 |       0.424 |    0.463 | 0.427 | 0.375-0.476 |       0   |
|       0 | bge-small: full Ardook description          |    0.512 |      0.507 |       0.489 |    0.507 | 0.496 | 0.443-0.544 |       0   |
|       1 | bge-small: theme-only description           |    0.579 |      0.51  |       0.478 |    0.51  | 0.487 | 0.432-0.539 |       0   |
|       1 | bge-small: best of 7 short queries          |    0.585 |      0.5   |       0.469 |    0.5   | 0.476 | 0.422-0.530 |       0   |
|       2 | TF-IDF + LogReg                             |    0.997 |      0.413 |       0.381 |    0.413 | 0.36  | 0.314-0.406 |       0.3 |
|       2 | MiniLM embeddings + LogReg                  |    0.967 |      0.567 |       0.582 |    0.567 | 0.56  | 0.508-0.612 |       0.1 |
|       2 | bge-small embeddings + LogReg               |    0.973 |      0.68  |       0.694 |    0.68  | 0.684 | 0.633-0.735 |       0.1 |
|       3 | MiniLM: 23 criterion scores + LogReg        |    0.823 |      0.647 |       0.657 |    0.647 | 0.645 | 0.591-0.696 |       0.1 |
|       3 | bge-small: 23 criterion scores + LogReg     |    0.813 |      0.74  |       0.757 |    0.74  | 0.74  | 0.689-0.787 |       0.1 |
|       4 | MiniLM: per-criterion classifiers + rule    |    0.987 |      0.577 |       0.626 |    0.577 | 0.568 | 0.512-0.620 |       0.3 |
|       4 | bge-small: per-criterion classifiers + rule |    0.997 |      0.647 |       0.67  |    0.647 | 0.649 | 0.590-0.702 |       0.3 |
|       5 | DistilBERT fine-tuned                       |    1     |      0.707 |       0.774 |    0.707 | 0.689 | 0.640-0.738 |     344.9 |

Best method of each round, chosen on validation:


|   round | method                                      |   val_f1 |   accuracy |   precision |   recall |    f1 | f1_95ci     |
|--------:|:--------------------------------------------|---------:|-----------:|------------:|---------:|------:|:------------|
|       0 | bge-small: full Ardook description          |    0.512 |      0.507 |       0.489 |    0.507 | 0.496 | 0.443-0.544 |
|       1 | MiniLM: theme-only description              |    0.596 |      0.46  |       0.429 |    0.46  | 0.43  | 0.379-0.476 |
|       2 | TF-IDF + LogReg                             |    0.997 |      0.413 |       0.381 |    0.413 | 0.36  | 0.314-0.406 |
|       3 | MiniLM: 23 criterion scores + LogReg        |    0.823 |      0.647 |       0.657 |    0.647 | 0.645 | 0.591-0.696 |
|       4 | bge-small: per-criterion classifiers + rule |    0.997 |      0.647 |       0.67  |    0.647 | 0.649 | 0.590-0.702 |
|       5 | DistilBERT fine-tuned                       |    1     |      0.707 |       0.774 |    0.707 | 0.689 | 0.640-0.738 |

In [13]:
# Final pick: highest validation F1. Rows of the matrix = true class, columns = predicted.
best = res.loc[res.val_f1.idxmax(), "method"]
print("Picked on validation:", best, "\n")
print(classification_report(y_test, predictions[best], target_names=LABELS, digits=3, zero_division=0))
pd.DataFrame(confusion_matrix(y_test, predictions[best]), index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS])

Picked on validation: DistilBERT fine-tuned 

              precision    recall  f1-score   support

         Low      0.957     0.890     0.922       100
    Moderate      0.539     0.900     0.674       100
        High      0.825     0.330     0.471       100

    accuracy                          0.707       300
   macro avg      0.774     0.707     0.689       300
weighted avg      0.774     0.707     0.689       300



,pred Low,pred Moderate,pred High
true Low,89,11,0
true Moderate,3,90,7
true High,1,66,33


### Did the round 0 failures improve?

In [14]:
low = test_df[test_df.label == "Low"]
high = test_df[test_df.label == "High"]
is_us = high.text.str.contains("US-based")
causes = {"theme": "theme", "geo": "geography", "entity": "entity"}
rows = {}
for name in res.method:
    pred = pd.Series(predictions[name], index=test_df.index)
    row = {}
    for col, desc in causes.items():
        mask = (low[col] == "bad") & (low[[c for c in causes if c != col]] != "bad").all(axis=1)
        row[f"Low: {desc} only (n={mask.sum()})"] = (pred[low[mask].index] == LABEL2ID["Low"]).mean()
    row[f"High: US (n={is_us.sum()})"] = (pred[high[is_us].index] == LABEL2ID["High"]).mean()
    row[f"High: other (n={(~is_us).sum()})"] = (pred[high[~is_us].index] == LABEL2ID["High"]).mean()
    rows[name] = row
display(Markdown(pd.DataFrame(rows).T.round(3).to_markdown()))

|                                             |   Low: theme only (n=24) |   Low: geography only (n=20) |   Low: entity only (n=17) |   High: US (n=32) |   High: other (n=68) |
|:--------------------------------------------|-------------------------:|-----------------------------:|--------------------------:|------------------:|---------------------:|
| MiniLM: full Ardook description             |                    0.833 |                         1    |                     0.412 |             0     |                0.603 |
| MiniLM: theme-only description              |                    0.958 |                         0.55 |                     0.471 |             0.594 |                0.485 |
| MiniLM: best of 7 short queries             |                    0.958 |                         0.5  |                     0.529 |             0.625 |                0.5   |
| bge-small: full Ardook description          |                    0.625 |                         0.75 |                     0.235 |             0     |                0.912 |
| bge-small: theme-only description           |                    0.917 |                         0.4  |                     0.412 |             0.844 |                0.632 |
| bge-small: best of 7 short queries          |                    1     |                         0.45 |                     0.412 |             0.688 |                0.603 |
| TF-IDF + LogReg                             |                    0.583 |                         0.7  |                     0.471 |             0.031 |                0.044 |
| MiniLM embeddings + LogReg                  |                    0.792 |                         0.25 |                     0.882 |             0.375 |                0.412 |
| bge-small embeddings + LogReg               |                    0.958 |                         0.1  |                     0.882 |             0.75  |                0.544 |
| MiniLM: 23 criterion scores + LogReg        |                    0.708 |                         0.1  |                     0.529 |             0.656 |                0.5   |
| bge-small: 23 criterion scores + LogReg     |                    0.875 |                         0.2  |                     0.588 |             0.781 |                0.618 |
| MiniLM: per-criterion classifiers + rule    |                    0.625 |                         0.4  |                     0.353 |             0.469 |                0.338 |
| bge-small: per-criterion classifiers + rule |                    0.667 |                         0.2  |                     0.412 |             0.656 |                0.588 |
| DistilBERT fine-tuned                       |                    0.958 |                         0.95 |                     0.471 |             0.5   |                0.25  |